# get dummyjson data

## 1 request dummyjson api

## 2 get data and save them on posgresql

In [103]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

my_params = {
    "limit" : 190
}
products = requests.get(os.getenv('PRODUCTS_API'), params=my_params)

In [104]:
import psycopg2

conn = psycopg2.connect(
    host=os.getenv('PGHOST'),
    port=os.getenv('PGPORT'),
    dbname=os.getenv('PGDATABASE'),
    user=os.getenv('PGUSER'),
    password=os.getenv('PGPASSWORD')
)

In [105]:
data = products.json()

In [106]:
main_keys = products.json()['products'][0]

multi_value = []
for key, value in products.json()['products'][0].items():
    if not isinstance(value, (int, str, float)):
        main_keys.pop(key)
        multi_value.append(key)
main_keys = main_keys.keys()

In [107]:
multi_value

['tags', 'dimensions', 'reviews', 'meta', 'images']

In [108]:
main_table = {
    'id': "SERIAL PRIMARY KEY",
    'title': "VARCHAR(100) NOT NULL",
    'description' : "text",
    'category' : "VARCHAR(100)",
    'price' : "NUMERIC",
    'discountPercentage' : "NUMERIC",
    'rating' : "NUMERIC",
    'stock' : "INT",
    'brand' : "VARCHAR(100)",
    'sku' : "VARCHAR(50) UNIQUE",
    'weight' : "NUMERIC",
    'warrantyInformation' : "VARCHAR(100)",
    'shippingInformation' : "VARCHAR(100)",
    'availabilityStatus' : "VARCHAR(100)",
    'returnPolicy' : "VARCHAR(100)",
    'minimumOrderQuantity' : "INT",
    'thumbnail' : "text"
}

In [109]:
parts = [f"{name} {dtype}"for name, dtype in main_table.items()]

query = f"CREATE TABLE IF NOT EXISTS dummy.products({', '.join(parts)});"

In [110]:
cur = conn.cursor()
try:
    cur.execute(f"""
    {query}
                """)
    conn.commit()
except Exception as e:
    conn.rollback()
    print('fail', e)

In [111]:
query_tags = "CREATE TABLE IF NOT EXISTS dummy.tags(product_id INTEGER NOT NULL REFERENCES dummy.products(id) ON DELETE CASCADE, tags VARCHAR(100), PRIMARY KEY (product_id, tags));"
query_dimensions = "CREATE TABLE IF NOT EXISTS dummy.dimensions(product_id INTEGER NOT NULL REFERENCES dummy.products(id) ON DELETE CASCADE, width NUMERIC, height NUMERIC, depth NUMERIC, PRIMARY KEY (product_id));"
query_reviews = "CREATE TABLE IF NOT EXISTS dummy.reviews(product_id INTEGER NOT NULL REFERENCES dummy.products(id) ON DELETE CASCADE, reviews JSONB, PRIMARY KEY (product_id));"
query_images = "CREATE TABLE IF NOT EXISTS dummy.images(product_id INTEGER NOT NULL REFERENCES dummy.products(id) ON DELETE CASCADE, images text, PRIMARY KEY (product_id));"

query_list = [query_tags, query_dimensions, query_reviews, query_images]

In [112]:
for key in multi_value:
    print(key, products.json()['products'][10][key])

tags ['furniture', 'beds']
dimensions {'width': 28.16, 'height': 25.36, 'depth': 17.28}
reviews [{'rating': 2, 'comment': 'Would not recommend!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Christopher West', 'reviewerEmail': 'christopher.west@x.dummyjson.com'}, {'rating': 4, 'comment': 'Highly impressed!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Vivian Carter', 'reviewerEmail': 'vivian.carter@x.dummyjson.com'}, {'rating': 1, 'comment': 'Poor quality!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Mason Wright', 'reviewerEmail': 'mason.wright@x.dummyjson.com'}]
meta {'createdAt': '2025-04-30T09:41:02.053Z', 'updatedAt': '2025-04-30T09:41:02.053Z', 'barcode': '3610757456581', 'qrCode': 'https://cdn.dummyjson.com/public/qr-code.png'}
images ['https://cdn.dummyjson.com/product-images/furniture/annibale-colombo-bed/1.webp', 'https://cdn.dummyjson.com/product-images/furniture/annibale-colombo-bed/2.webp', 'https://cdn.dummyjson.com/product-images/furniture/anni

In [113]:
for query in query_list:
    try:
        cur.execute(f"""
        {query}
                    """)
        conn.commit()
    except Exception as e:
        conn.rollback()
        print('FAIL', e)

In [114]:
query_insert_product = """
INSERT INTO dummy.products(
id, title, description, category, price, discountpercentage,
rating, stock, brand, sku,
weight, warrantyinformation, shippinginformation,
availabilitystatus, returnPolicy, minimumOrderQuantity, thumbnail
) VALUES (
%s, %s, %s, %s, %s,
%s, %s, %s, %s, %s,
%s, %s, %s,
%s, %s, %s, %s
)
ON CONFLICT (id) DO UPDATE SET
    title = EXCLUDED.title,
    description = EXCLUDED.description,
    price = EXCLUDED.price,
    stock = EXCLUDED.stock,
    brand = EXCLUDED.brand,
    category = EXCLUDED.category,
    discountpercentage = EXCLUDED.discountpercentage,
    rating = EXCLUDED.rating,
    sku = EXCLUDED.sku,
    weight = EXCLUDED.weight,
    warrantyinformation = EXCLUDED.warrantyinformation,
    shippinginformation = EXCLUDED.shippinginformation,
    availabilitystatus = EXCLUDED.availabilitystatus,
    returnPolicy = EXCLUDED.returnPolicy,
    minimumOrderQuantity = EXCLUDED.minimumOrderQuantity,
    thumbnail = EXCLUDED.thumbnail;
"""

In [115]:
for item in data['products']:
    values = (
        item.get('id'),                                    # id
        item.get('title', 'بدون عنوان'),                     # title
        item.get('description', ''),                       # description
        item.get('category', ''),                          # category
        item.get('price', 0),                              # price
        item.get('discountPercentage', 0),                 # discountpercentage
        item.get('rating', 0),                             # rating
        item.get('stock', 0),                              # stock
        item.get('brand', ''),                             # brand
        item.get('sku', ''),                               # sku
        item.get('weight', 0),                             # weight
        item.get('warrantyInformation', ''),               # warrantyinformation
        item.get('shippingInformation', ''),               # shippinginformation
        item.get('availabilityStatus', ''),                # availabilitystatus
        item.get('returnPolicy', ''),                      # returnpolicy
        item.get('minimumOrderQuantity', 0),               # minimumorderquantity
        item.get('thumbnail', '')                          # thumbnail
    )
    try:
        cur.execute(query_insert_product, values)
        conn.commit()
        print("succesful")
    except Exception as e:
        conn.rollback()
        print("fail", e)

succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful


In [116]:
query_tags_insert = """
INSERT INTO dummy.tags(product_id, tags)
VALUES (%s, %s)
ON CONFLICT (product_id, tags) DO UPDATE SET
    tags = EXCLUDED.tags
"""

In [117]:
for product in data['products']:
        product_id = product.get("id")
        for tag in product.get("tags"):
            values = (product_id, tag)

            try:
                cur.execute(query_tags_insert, values)
                conn.commit()
                print("succesful")
            except Exception as e:
                conn.rollback()
                print("fail", e)

succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful


In [118]:
query_dimention_insert = """
INSERT INTO dummy.dimensions(product_id, width, height, depth)
VALUES(%s, %s, %s, %s)
ON CONFLICT (product_id) DO UPDATE SET
    width = EXCLUDED.width,
    height = EXCLUDED.height,
    depth = EXCLUDED.depth
"""

In [119]:
for product in data["products"]:
    product_id = product.get("id")
    dimensions = product.get("dimensions")
    if dimensions is None:
        values = (product_id, 0, 0, 0)
    else:
        width = dimensions.get("width", 0)
        height = dimensions.get("height", 0)
        depth = dimensions.get("depth", 0)
        values = (product_id, width, height, depth)
    try:
        cur.execute(query_dimention_insert, values)
        conn.commit()
        print("succesful")
    except Exception as e :
        conn.rollback()
        print("fail", e)

succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful


In [120]:
query_reviews_insert = """
INSERT INTO dummy.reviews (product_id, reviews)
VALUES (%s, %s::jsonb)
ON CONFLICT (product_id) DO UPDATE SET
    reviews = EXCLUDED.reviews
"""

In [121]:
import json

products = data["products"]

for product in products:
    review = product.get("reviews", "")
    json_review = json.dumps(review)
    product_id = product["id"]
    values = (product_id, json_review)

    try:
        cur.execute(query_reviews_insert, values)
        conn.commit()
        print("succesful")

    except Exception as e:
        conn.rollback()
        print("fail", e)

succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful


In [122]:
query_images_insert = """
INSERT INTO dummy.images(product_id, images)
VALUES (%s, %s)
ON CONFLICT (product_id) DO UPDATE SET
    images = EXCLUDED.images
"""

In [125]:
for product in products:
    product_id = product["id"]
    all_images = "".join(product["images"])
    values = (product_id, all_images)

    try:
        cur.execute(query_images_insert, values)
        conn.commit()
        print("succesful")

    except Exception as e:
        conn.rollback()
        print("fail", e)

succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful
succesful


In [126]:
cur.close()
conn.close()